In [26]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv('climate.csv',index_col=['Date Time'])

In [3]:
data.head()

,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
Date Time,,,,,,,,,,,,,,
01.01.2009 00:10:00,996.52,-8.02,265.40,-8.90,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
01.01.2009 00:20:00,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.80,0.72,1.50,136.1
01.01.2009 00:30:00,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.20,1.88,3.02,1310.24,0.19,0.63,171.6
01.01.2009 00:40:00,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.50,198.0
01.01.2009 00:50:00,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.00,0.32,0.63,214.3


In [9]:
data.index = pd.to_datetime(data.index,format='%d.%m.%Y %H:%M:%S')

In [18]:
data = data[(data.index.minute == 0) & (data.index.hour % 3 == 0)]

In [20]:
tempc = data['T (degC)']
var = data.drop(columns=['T (degC)','Tpot (K)'])

In [22]:
var

,p (mbar),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
Date Time,,,,,,,,,,,,
2009-01-01 03:00:00,996.84,-9.66,93.50,3.13,2.93,0.20,1.83,2.94,1312.18,0.18,0.63,167.2
2009-01-01 06:00:00,997.71,-10.62,92.70,2.93,2.71,0.21,1.69,2.72,1317.71,0.05,0.50,146.0
2009-01-01 09:00:00,999.69,-8.84,91.20,3.43,3.13,0.30,1.95,3.13,1310.14,0.34,0.63,202.2
2009-01-01 12:00:00,1000.30,-8.28,89.60,3.64,3.27,0.38,2.03,3.26,1306.98,1.84,2.63,184.4
2009-01-01 15:00:00,999.88,-7.00,90.40,3.99,3.61,0.38,2.25,3.61,1300.51,1.17,1.88,134.9
...,...,...,...,...,...,...,...,...,...,...,...,...
2016-12-31 12:00:00,1004.76,-5.90,70.20,5.60,3.93,1.67,2.44,3.91,1285.09,0.86,1.56,211.2
2016-12-31 15:00:00,1003.12,-3.09,55.34,8.78,4.86,3.92,3.02,4.85,1253.58,0.30,0.70,134.1
2016-12-31 18:00:00,1002.27,-4.90,69.81,6.07,4.24,1.83,2.64,4.23,1276.52,0.39,1.04,220.7


In [23]:
from sklearn.preprocessing import StandardScaler

In [24]:
scaler = StandardScaler()
var = scaler.fit_transform(var)

In [27]:
seq_len = 56
step = 2

In [32]:
sequences = []
targets = []

for i in range(0,len(data)-seq_len-7,step):
    seq = var[i:i+seq_len]
    target = tempc[i+seq_len+7]
    sequences.append(seq)
    targets.append(target)

sequences, targets = np.array(sequences), np.array(targets)

In [33]:
sequences.shape

(11650, 56, 12)

In [42]:
X_train, X_val, X_test = sequences[:8000], sequences[8000:9500], sequences[9500:]
y_train, y_val, y_test = targets[:8000], targets[8000:9500], targets[9500:]

In [36]:
import tensorflow as tf
from tensorflow.keras.layers import GRU,Dense,Dropout
from tensorflow.keras.models import Sequential

In [37]:
model = Sequential()
model.add(GRU(units=128,activation='tanh',recurrent_dropout=0.3,input_shape=(56,12),return_sequences=True))
model.add(GRU(units=128,activation='tanh',recurrent_dropout=0.3,dropout=0.3))
model.add(Dropout(0.5))
model.add(Dense(128,activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1,activation='linear'))

In [38]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 gru (GRU)                   (None, 56, 128)           54528     
                                                                 
 gru_1 (GRU)                 (None, 128)               99072     
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense (Dense)               (None, 128)               16512     
                                                                 
 dropout_1 (Dropout)         (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 1)                 129       
                                                                 
Total params: 170241 (665.00 KB)
Trainable params: 17024

In [39]:
opt = tf.optimizers.Adam(learning_rate=0.001)
loss = tf.losses.MeanAbsoluteError()
model.compile(optimizer=opt,loss=loss)

In [40]:
model.fit(X_train,y_train,batch_size=50,epochs=20,validation_data=(X_val,y_val),verbose=2)

Epoch 1/20
160/160 - 15s - loss: 4.3910 - val_loss: 2.5138 - 15s/epoch - 97ms/step
Epoch 2/20
160/160 - 13s - loss: 3.1559 - val_loss: 2.5252 - 13s/epoch - 81ms/step
Epoch 3/20
160/160 - 14s - loss: 3.0037 - val_loss: 2.5559 - 14s/epoch - 85ms/step
Epoch 4/20
160/160 - 14s - loss: 2.9401 - val_loss: 2.4510 - 14s/epoch - 87ms/step
Epoch 5/20
160/160 - 14s - loss: 2.9330 - val_loss: 2.7500 - 14s/epoch - 90ms/step
Epoch 6/20
160/160 - 14s - loss: 2.9156 - val_loss: 2.6245 - 14s/epoch - 90ms/step
Epoch 7/20
160/160 - 15s - loss: 2.9004 - val_loss: 2.4389 - 15s/epoch - 91ms/step
Epoch 8/20
160/160 - 15s - loss: 2.8561 - val_loss: 2.3904 - 15s/epoch - 97ms/step
Epoch 9/20
160/160 - 16s - loss: 2.8475 - val_loss: 2.5190 - 16s/epoch - 98ms/step
Epoch 10/20
160/160 - 16s - loss: 2.8149 - val_loss: 2.3737 - 16s/epoch - 97ms/step
Epoch 11/20
160/160 - 15s - loss: 2.8719 - val_loss: 2.4871 - 15s/epoch - 91ms/step
Epoch 12/20
160/160 - 15s - loss: 2.8078 - val_loss: 2.3930 - 15s/epoch - 94ms/step
E

In [43]:
model.evaluate(X_test,y_test)

68/68 [==============================] - 1s 10ms/step - loss: 2.3897


2.3896737098693848